In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module3_uturn'
MODEL_DIR = ROOT / 'models' / 'module3' / 'baselines'
MET_DIR = ROOT / 'metrics' / 'module3'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'module3_ml_table.parquet')
feat_cols = json.loads((PROC / 'module3_ml_meta.json').read_text())['features']
X = df[feat_cols].fillna(df[feat_cols].median(numeric_only=True))
y = df['y'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, y_train.value_counts().to_dict())

Mounted at /content/drive
(877, 16) {1: 489, 0: 388}


In [2]:
!pip -q install xgboost joblib

In [3]:
def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    metrics = {
        'model': name, 'category': 'Base ML', 'module': 'module3_uturn',
        'accuracy': float(accuracy_score(y_te, pred)),
        'precision': float(precision_score(y_te, pred, average='binary', zero_division=0)),
        'recall': float(recall_score(y_te, pred, average='binary', zero_division=0)),
        'f1': float(f1_score(y_te, pred, average='binary', zero_division=0)),
    }
    try:
        metrics['roc_auc'] = float(roc_auc_score(y_te, model.predict_proba(X_te)[:,1]))
    except Exception:
        metrics['roc_auc'] = None
    print('\n', name)
    print(classification_report(y_te, pred, digits=4, zero_division=0))
    print(metrics)
    return metrics

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])
pipe_lr.fit(X_train, y_train)
m0 = eval_model('Logistic_Regression', pipe_lr, X_test, y_test)
joblib.dump(pipe_lr, MODEL_DIR/'logistic_regression.joblib')
(MET_DIR/'Logistic_Regression.json').write_text(json.dumps(m0, indent=2))

rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=2,
    class_weight='balanced_subsample', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
m1 = eval_model('Random_Forest', rf, X_test, y_test)
joblib.dump(rf, MODEL_DIR/'random_forest.joblib')
(MET_DIR/'Random_Forest.json').write_text(json.dumps(m1, indent=2))

vc = y_train.value_counts()
w_map = {c: len(y_train)/(len(vc)*cnt) for c,cnt in vc.items()}
sw = y_train.map(w_map)
gb = GradientBoostingClassifier(random_state=42, max_depth=3, learning_rate=0.08, n_estimators=250)
gb.fit(X_train, y_train, sample_weight=sw)
m2 = eval_model('Gradient_Boosting', gb, X_test, y_test)
joblib.dump(gb, MODEL_DIR/'gradient_boosting.joblib')
(MET_DIR/'Gradient_Boosting.json').write_text(json.dumps(m2, indent=2))

from xgboost import XGBClassifier
spw = (y_train==0).sum() / max((y_train==1).sum(), 1)
xgb = XGBClassifier(
    n_estimators=350, max_depth=5, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9,
    objective='binary:logistic', eval_metric='auc',
    scale_pos_weight=spw, tree_method='hist',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
m3 = eval_model('XGBoost', xgb, X_test, y_test)
joblib.dump(xgb, MODEL_DIR/'xgboost.joblib')
(MET_DIR/'XGBoost.json').write_text(json.dumps(m3, indent=2))

summary = [m0,m1,m2,m3]
pd.DataFrame(summary).to_csv(MET_DIR/'baselines_summary.csv', index=False)
(MET_DIR/'baselines_summary.json').write_text(json.dumps(summary, indent=2))
print(pd.DataFrame(summary)[['model','accuracy','precision','recall','f1','roc_auc']])
print('STEP 9a OK')


 Logistic_Regression
              precision    recall  f1-score   support

           0     1.0000    0.9794    0.9896        97
           1     0.9840    1.0000    0.9919       123

    accuracy                         0.9909       220
   macro avg     0.9920    0.9897    0.9908       220
weighted avg     0.9911    0.9909    0.9909       220

{'model': 'Logistic_Regression', 'category': 'Base ML', 'module': 'module3_uturn', 'accuracy': 0.990909090909091, 'precision': 0.984, 'recall': 1.0, 'f1': 0.9919354838709677, 'roc_auc': 0.999580923644288}

 Random_Forest
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        97
           1     1.0000    1.0000    1.0000       123

    accuracy                         1.0000       220
   macro avg     1.0000    1.0000    1.0000       220
weighted avg     1.0000    1.0000    1.0000       220

{'model': 'Random_Forest', 'category': 'Base ML', 'module': 'module3_uturn', 'accuracy': 1.0, 'precisio

In [1]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted')

Mounted at /content/drive


In [2]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
print((ROOT/'datasets/processed/module3_uturn/module3_ml_table.parquet').exists())
print(list((ROOT/'models/module3/baselines').glob('*.joblib')))

True
[PosixPath('/content/drive/MyDrive/TrafficAI_ControlCenter/models/module3/baselines/logistic_regression.joblib'), PosixPath('/content/drive/MyDrive/TrafficAI_ControlCenter/models/module3/baselines/random_forest.joblib'), PosixPath('/content/drive/MyDrive/TrafficAI_ControlCenter/models/module3/baselines/gradient_boosting.joblib'), PosixPath('/content/drive/MyDrive/TrafficAI_ControlCenter/models/module3/baselines/xgboost.joblib')]


In [3]:
import torch, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

assert torch.cuda.is_available(), 'Enable T4'
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module3_uturn'
MODEL_DIR = ROOT / 'models' / 'module3' / 'dl'
MET_DIR = ROOT / 'metrics' / 'module3'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)

labels = pd.read_csv(PROC / 'uturn_candidate_labels.csv')[['vehicle_id', 'uturn_candidate']]
tracks = pd.read_parquet(PROC / 'ngsim_tracks_sample.parquet').sort_values(['vehicle_id', 'frame_id'])

# sequence feature columns available in tracks
seq_feats = [c for c in ['x','y','speed','dx','dy','heading','heading_delta'] if c in tracks.columns]
print('seq_feats', seq_feats)

SEQ_LEN = 32
Xs, ys = [], []
lab_map = dict(zip(labels['vehicle_id'], labels['uturn_candidate'].astype(int)))

for vid, g in tracks.groupby('vehicle_id', sort=False):
    if vid not in lab_map:
        continue
    g = g.copy()
    arr = g[seq_feats].to_numpy(np.float32)
    if len(arr) < 8:
        continue
    # pad / truncate to SEQ_LEN
    if len(arr) >= SEQ_LEN:
        # use last SEQ_LEN steps (near maneuver end)
        window = arr[-SEQ_LEN:]
    else:
        pad = np.repeat(arr[:1], SEQ_LEN - len(arr), axis=0)
        window = np.vstack([pad, arr])
    Xs.append(window)
    ys.append(lab_map[vid])

X = np.stack(Xs).astype(np.float32)
y = np.array(ys, dtype=np.int64)
print('sequences', X.shape, 'pos', y.sum(), 'neg', (y==0).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
mean = X_train.mean((0,1), keepdims=True)
std = X_train.std((0,1), keepdims=True) + 1e-6
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
np.save(MODEL_DIR / 'seq_mean.npy', mean)
np.save(MODEL_DIR / 'seq_std.npy', std)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X); self.y = torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(SeqDS(X_train, y_train), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SeqDS(X_test, y_test), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

neg, pos = (y_train==0).sum(), max((y_train==1).sum(), 1)
pos_weight = torch.tensor([neg/pos], dtype=torch.float32, device=device)
print('pos_weight', float(pos_weight))
F = len(seq_feats)

Tesla T4
seq_feats ['x', 'y', 'speed', 'dx', 'dy', 'heading', 'heading_delta']
sequences (1098, 32, 7) pos 612 neg 486
pos_weight 0.7955009937286377


In [4]:
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.w = nn.Linear(d, 1)
    def forward(self, h):
        a = torch.softmax(self.w(h), dim=1)
        return (a * h).sum(1)

def evaluate(model):
    model.eval(); preds=[]; trues=[]; probs=[]
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logit = model(xb).squeeze(1)
            prob = torch.sigmoid(logit).cpu().numpy()
            pred = (prob >= 0.5).astype(int)
            preds.append(pred); trues.append(yb.numpy()); probs.append(prob)
    yt, yp, pr = np.concatenate(trues), np.concatenate(preds), np.concatenate(probs)
    return {
        'accuracy': float(accuracy_score(yt, yp)),
        'precision': float(precision_score(yt, yp, zero_division=0)),
        'recall': float(recall_score(yt, yp, zero_division=0)),
        'f1': float(f1_score(yt, yp, zero_division=0)),
        'roc_auc': float(roc_auc_score(yt, pr)),
    }

def train_model(model, name, epochs=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_f1, best = -1, None
    for ep in range(1, epochs+1):
        model.train(); total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.float().to(device)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(1), yb)
            loss.backward(); opt.step()
            total += loss.item() * len(yb)
        met = evaluate(model)
        print(f'{name} epoch {ep}/{epochs} loss={total/len(y_train):.4f} f1={met["f1"]:.4f} auc={met["roc_auc"]:.4f}')
        if met['f1'] > best_f1:
            best_f1 = met['f1']
            best = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best)
    met = evaluate(model)
    met.update({'model': name, 'category': 'Optimized DL', 'module': 'module3_uturn'})
    torch.save({'state_dict': best, 'seq_feats': seq_feats, 'seq_len': SEQ_LEN}, MODEL_DIR / f'{name}.pt')
    (MET_DIR / f'{name}.json').write_text(json.dumps(met, indent=2))
    print('Saved', name, met)
    return met

In [5]:
class LSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(F, 64, batch_first=True)
        self.attn = AttnPool(64); self.fc = nn.Linear(64, 1)
    def forward(self, x):
        h,_ = self.lstm(x); return self.fc(self.attn(h))

class BiLSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(F, 64, batch_first=True, bidirectional=True)
        self.attn = AttnPool(128); self.fc = nn.Linear(128, 1)
    def forward(self, x):
        h,_ = self.lstm(x); return self.fc(self.attn(h))

class TCNAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(F, 64, 3, padding=1)
        self.c2 = nn.Conv1d(64, 64, 3, padding=1)
        self.c3 = nn.Conv1d(64, 64, 3, padding=1)
        self.attn = AttnPool(64); self.fc = nn.Linear(64, 1)
    def forward(self, x):
        z = x.transpose(1,2)
        z = torch.relu(self.c3(torch.relu(self.c2(torch.relu(self.c1(z)))))).transpose(1,2)
        return self.fc(self.attn(z))

class FeatureAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(F, 64), nn.ReLU(), nn.Linear(64, F), nn.Sigmoid())
        self.proj = nn.Linear(F, 64)
        layer = nn.TransformerEncoderLayer(64, 4, 128, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 2)
        self.attn = AttnPool(64); self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = x * self.feat(x[:, -1, :]).unsqueeze(1)
        return self.fc(self.attn(self.enc(self.proj(x))))

class TransformerCW(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(F, 64)
        layer = nn.TransformerEncoderLayer(64, 4, 128, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 3)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        return self.fc(self.enc(self.proj(x))[:, -1, :])

class GATGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_proj = nn.Linear(F, 32)
        self.attn_l = nn.Linear(64, 1)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        b,t,_ = x.shape
        h = torch.relu(self.in_proj(x))
        outs = []
        idx = torch.arange(t, device=x.device).float()
        for i in range(t):
            center = h[:, i:i+1, :].expand(-1, t, -1)
            score = self.attn_l(torch.cat([center, h], -1)).squeeze(-1)
            score = score - ((idx - i).abs().unsqueeze(0) / max(t-1, 1))
            a = torch.softmax(score, 1).unsqueeze(-1)
            outs.append((a * h).sum(1))
        g = torch.stack(outs, 1)
        y,_ = self.gru(g)
        return self.fc(y[:, -1, :])

class PolicyHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BiLSTMAttn()
        self.backbone.fc = nn.Identity()
        self.policy = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x):
        return self.policy(self.backbone(x))

In [6]:
results = []
results.append(train_model(LSTMAttn(), 'LSTM_Attention', epochs=10))
results.append(train_model(BiLSTMAttn(), 'BiLSTM_Attention', epochs=10))
results.append(train_model(TCNAttn(), 'TCN_Attention', epochs=10))
results.append(train_model(FeatureAttn(), 'Transformer_Feature_Attention', epochs=10))
results.append(train_model(TransformerCW(), 'Transformer_Encoder_ClassWeighted', epochs=10))
results.append(train_model(GATGRU(), 'GAT_GRU', epochs=8))
results.append(train_model(PolicyHead(), 'Decision_Policy_Head', epochs=10))

sum_df = pd.DataFrame(results)[['model','category','accuracy','precision','recall','f1','roc_auc']]
print(sum_df.sort_values('f1', ascending=False))
sum_df.to_csv(MET_DIR / 'dl_summary.csv', index=False)
(MET_DIR / 'dl_summary.json').write_text(json.dumps(results, indent=2))

base = json.loads((MET_DIR / 'baselines_summary.json').read_text())
arena = base + results
pd.DataFrame(arena).to_csv(MET_DIR / 'arena_module3_11models.csv', index=False)
(MET_DIR / 'arena_module3_11models.json').write_text(json.dumps(arena, indent=2))

# verify saves
print('dl pts:', list(MODEL_DIR.glob('*.pt')))
print('STEP 9b OK — Module 3 eleven-model arena saved')

ValueError: Input contains NaN.

In [7]:
import numpy as np

# replace NaN/Inf in sequences
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

# rebuild loaders
train_loader = DataLoader(SeqDS(X_train, y_train), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SeqDS(X_test, y_test), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print('NaN train', np.isnan(X_train).sum(), 'NaN test', np.isnan(X_test).sum())

NaN train 0 NaN test 0


In [8]:
def evaluate(model):
    model.eval(); preds=[]; trues=[]; probs=[]
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
            logit = model(xb).squeeze(1)
            logit = torch.nan_to_num(logit, nan=0.0, posinf=50.0, neginf=-50.0)
            prob = torch.sigmoid(logit).cpu().numpy()
            prob = np.nan_to_num(prob, nan=0.5)
            pred = (prob >= 0.5).astype(int)
            preds.append(pred); trues.append(yb.numpy()); probs.append(prob)
    yt = np.concatenate(trues)
    yp = np.concatenate(preds)
    pr = np.concatenate(probs)
    out = {
        'accuracy': float(accuracy_score(yt, yp)),
        'precision': float(precision_score(yt, yp, zero_division=0)),
        'recall': float(recall_score(yt, yp, zero_division=0)),
        'f1': float(f1_score(yt, yp, zero_division=0)),
    }
    try:
        out['roc_auc'] = float(roc_auc_score(yt, pr))
    except Exception:
        out['roc_auc'] = None
    return out

In [9]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
            logit = model(xb).squeeze(1)
            logit = torch.nan_to_num(logit, nan=0.0, posinf=50.0, neginf=-50.0)
            prob = torch.sigmoid(logit).cpu().numpy()
            prob = np.nan_to_num(prob, nan=0.5)
            pred = (prob >= 0.5).astype(int)
            preds.append(pred)
            trues.append(yb.numpy())
            probs.append(prob)
    yt = np.concatenate(trues)
    yp = np.concatenate(preds)
    pr = np.concatenate(probs)
    out = {
        'accuracy': float(accuracy_score(yt, yp)),
        'precision': float(precision_score(yt, yp, zero_division=0)),
        'recall': float(recall_score(yt, yp, zero_division=0)),
        'f1': float(f1_score(yt, yp, zero_division=0)),
    }
    try:
        out['roc_auc'] = float(roc_auc_score(yt, pr))
    except Exception:
        out['roc_auc'] = None
    return out

def train_model(model, name, epochs=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_f1, best = -1, None

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
            yb = yb.float().to(device)

            opt.zero_grad()
            loss = crit(model(xb).squeeze(1), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)

        met = evaluate(model)
        print(f'{name} epoch {ep}/{epochs} loss={total/len(y_train):.4f} f1={met["f1"]:.4f} auc={met["roc_auc"]}')
        if met['f1'] > best_f1:
            best_f1 = met['f1']
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best)
    met = evaluate(model)
    met.update({'model': name, 'category': 'Optimized DL', 'module': 'module3_uturn'})
    torch.save({'state_dict': best, 'seq_feats': seq_feats, 'seq_len': SEQ_LEN}, MODEL_DIR / f'{name}.pt')
    (MET_DIR / f'{name}.json').write_text(json.dumps(met, indent=2))
    print('Saved', name, met)
    return met

In [10]:
import json
import pandas as pd

results = []
results.append(train_model(LSTMAttn(), 'LSTM_Attention', epochs=10))
results.append(train_model(BiLSTMAttn(), 'BiLSTM_Attention', epochs=10))
results.append(train_model(TCNAttn(), 'TCN_Attention', epochs=10))
results.append(train_model(FeatureAttn(), 'Transformer_Feature_Attention', epochs=10))
results.append(train_model(TransformerCW(), 'Transformer_Encoder_ClassWeighted', epochs=10))
results.append(train_model(GATGRU(), 'GAT_GRU', epochs=8))
results.append(train_model(PolicyHead(), 'Decision_Policy_Head', epochs=10))

sum_df = pd.DataFrame(results)[['model', 'category', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
print(sum_df.sort_values('f1', ascending=False))
sum_df.to_csv(MET_DIR / 'dl_summary.csv', index=False)
(MET_DIR / 'dl_summary.json').write_text(json.dumps(results, indent=2))

base = json.loads((MET_DIR / 'baselines_summary.json').read_text())
arena = base + results
pd.DataFrame(arena).to_csv(MET_DIR / 'arena_module3_11models.csv', index=False)
(MET_DIR / 'arena_module3_11models.json').write_text(json.dumps(arena, indent=2))

print('dl pts:', list(MODEL_DIR.glob('*.pt')))
print('STEP 9b OK — Module 3 eleven-model arena saved')

LSTM_Attention epoch 1/10 loss=0.6107 f1=0.6735 auc=0.5404408683262091
LSTM_Attention epoch 2/10 loss=0.6055 f1=0.6312 auc=0.539602715614785
LSTM_Attention epoch 3/10 loss=0.6008 f1=0.6308 auc=0.5489900259827339
LSTM_Attention epoch 4/10 loss=0.5984 f1=0.6442 auc=0.5531807895398541
LSTM_Attention epoch 5/10 loss=0.5973 f1=0.6357 auc=0.5629871762635152
LSTM_Attention epoch 6/10 loss=0.5942 f1=0.6167 auc=0.5758947280194452
LSTM_Attention epoch 7/10 loss=0.5891 f1=0.6038 auc=0.6050624423770011
LSTM_Attention epoch 8/10 loss=0.5605 f1=0.6695 auc=0.6578660631967145
LSTM_Attention epoch 9/10 loss=0.5344 f1=0.6226 auc=0.7207275165535161
LSTM_Attention epoch 10/10 loss=0.5001 f1=0.6957 auc=0.7209789623669431
Saved LSTM_Attention {'accuracy': 0.6818181818181818, 'precision': 0.7476635514018691, 'recall': 0.6504065040650406, 'f1': 0.6956521739130435, 'roc_auc': 0.7209789623669431, 'model': 'LSTM_Attention', 'category': 'Optimized DL', 'module': 'module3_uturn'}
BiLSTM_Attention epoch 1/10 loss=0